# SocraticLoop system feasibility study

This notebook runs systems B and C on the executable eight-case benchmark:
- B: Qwen2.5-Coder-3B-Instruct without the SocraticLoop harness
- C: the same model with deterministic validation, checkpoint context, and safety policy

The Codex reference (A) uses `scripts/run_codex_reference.py` in a normal PowerShell environment. The post-trained system (D) requires a separately trained PEFT adapter and is not simulated here.

In [ ]:
REPO_URL = "https://github.com/jasperrrliu-ui/AC215-Project-SocraticLoop.git"
REPO_DIR = "AC215-Project-SocraticLoop"
MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"

In [ ]:
!rm -rf $REPO_DIR
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip -q install -U transformers accelerate bitsandbytes peft

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "scripts/prepare_agent_benchmark.py"], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/run_qwen_agent_benchmark.py", "--system", "qwen_raw", "--model", MODEL], check=True)
subprocess.run([sys.executable, "scripts/score_agent_benchmark.py", "--system", "qwen_raw", "--responses", "outputs/agent_benchmark/responses/qwen_raw.jsonl"], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/run_qwen_agent_benchmark.py", "--system", "qwen_harness", "--model", MODEL], check=True)
subprocess.run([sys.executable, "scripts/score_agent_benchmark.py", "--system", "qwen_harness", "--responses", "outputs/agent_benchmark/responses/qwen_harness.jsonl"], check=True)

In [ ]:
from pathlib import Path
import json
for report in sorted(Path("outputs/agent_benchmark/reports").glob("*_report.json")):
    payload = json.loads(report.read_text())
    print(f"\n=== {report.name} ===")
    print(json.dumps(payload["summary"], indent=2))

In [ ]:
!zip -qr agent-benchmark-results.zip outputs/agent_benchmark/responses outputs/agent_benchmark/reports
from google.colab import files
files.download("agent-benchmark-results.zip")